In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , classification_report , precision_recall_fscore_support
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestClassifier
import warnings
import os


In [2]:
from IPython.display import clear_output

In [79]:
a1 = pd.read_excel('dataset/case_study1.xlsx')
a2 = pd.read_excel('dataset/case_study2.xlsx')

In [4]:
df1 = a1.copy()
df2 = a2.copy()

In [5]:
# considering -99999 as NULL value
# remove nulls values

df1 = df1.loc[df1['Age_Oldest_TL'] != -99999]

In [6]:
columns_to_be_removed = []

for i in df2.columns :
    if df2.loc[df2[i] == -99999].shape[0] >10000:
        columns_to_be_removed.append(i)

In [7]:
columns_to_be_removed

['time_since_first_deliquency',
 'time_since_recent_deliquency',
 'max_delinquency_level',
 'max_deliq_6mts',
 'max_deliq_12mts',
 'CC_utilization',
 'PL_utilization',
 'max_unsec_exposure_inPct']

In [8]:
df2 = df2.drop(columns_to_be_removed , axis = 1)

In [9]:

for i in df2.columns:
    df2 = df2.loc[ df2[i] != -99999 ]

In [10]:
# df2.columns

In [11]:
# checking common columns names

for i in list(df1.columns):
    if i in list(df2.columns):
        print(i)

PROSPECTID


In [12]:
# merging both dataframes , using inner join so that no nulls are preset

df = pd.merge( df1 , df2 , how = 'inner' , left_on = ['PROSPECTID'] ,
              right_on = ['PROSPECTID'])

In [13]:
# checking categorcial columns :
cat_cols = []
for i in df.columns :
    if df[i].dtype == 'object':
        cat_cols.append(i)
        print(i)

MARITALSTATUS
EDUCATION
GENDER
last_prod_enq2
first_prod_enq2
Approved_Flag


In [14]:
# chi- square test

for i in cat_cols:
    if i != 'Approved_Flag':
        chi2 , pval , _ , _ = chi2_contingency(pd.crosstab(
            df[i] , 
            df['Approved_Flag']))
        print(f'For feature : {i} \n chi2_value : {chi2} \n pval : {pval}')

For feature : MARITALSTATUS 
 chi2_value : 1076.9871387543772 
 pval : 3.578180861038862e-233
For feature : EDUCATION 
 chi2_value : 187.81675366240617 
 pval : 2.6942265249737532e-30
For feature : GENDER 
 chi2_value : 24.56031272141628 
 pval : 1.907936100186563e-05
For feature : last_prod_enq2 
 chi2_value : 2444.9571510235596 
 pval : 0.0
For feature : first_prod_enq2 
 chi2_value : 1387.5609151031795 
 pval : 7.84997610555419e-287


 Since all the categorical features have pval <= 0.05 , we will accept all

In [15]:
# VIF for numerical features 

numeric_cols = []
for i in df.columns:
    if df[i].dtype != 'object' and i not in ['PROSPECTID' , 'Approved_Flag']:
        numeric_cols.append(i)
    

In [16]:
len(numeric_cols)

72

In [17]:
# VIF sequenctially check 

vif_data = df[numeric_cols]
total_columns = vif_data.shape[1]
columns_to_be_kept = []
column_index = 0

In [18]:
for i in range (0,total_columns):
    
    vif_value = variance_inflation_factor(vif_data, column_index)
    print (column_index,'---',vif_value)
    
    
    if vif_value <= 6:
        columns_to_be_kept.append( numeric_cols[i] )
        column_index = column_index+1
    
    else:
        vif_data = vif_data.drop([ numeric_cols[i] ] , axis=1)
clear_output()

In [19]:
len(columns_to_be_kept)

39

In [21]:
# Anova check on numerical :

from scipy.stats import f_oneway

columns_to_be_kept_numerical = []

for i in columns_to_be_kept:
    a = list(df[i])
    b = list(df['Approved_Flag'])

    group_P1 = [value for value , group in zip(a ,b) if group =='P1']
    group_P2 = [value for value , group in zip(a ,b) if group =='P2']
    group_P3 = [value for value , group in zip(a ,b) if group =='P3']
    group_P4 = [value for value , group in zip(a ,b) if group =='P4']

    f_stats , pval = f_oneway(group_P1 , group_P2 , group_P3 , group_P4)

    if pval <= 0.05:
        columns_to_be_kept_numerical.append(i)

 features selection has be done

In [22]:
# listing all the final features

features = columns_to_be_kept_numerical + ['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']

df = df[features + ['Approved_Flag']]

In [24]:
# label encoding the categorical features

# oridinal feature - EDUCATION
# SSC            : 1
# 12TH           : 2
# GRADUATE       : 3
# UNDER GRADUATE : 3
# POST-GRADUATE  : 4
# OTHERS         : 1
# PROFESSIONAL   : 3


df.loc[df['EDUCATION'] == 'SSC',['EDUCATION']]              = 1
df.loc[df['EDUCATION'] == '12TH',['EDUCATION']]             = 2
df.loc[df['EDUCATION'] == 'GRADUATE',['EDUCATION']]         = 3
df.loc[df['EDUCATION'] == 'UNDER GRADUATE',['EDUCATION']]   = 3
df.loc[df['EDUCATION'] == 'POST-GRADUATE',['EDUCATION']]    = 4
df.loc[df['EDUCATION'] == 'OTHERS',['EDUCATION']]           = 1
df.loc[df['EDUCATION'] == 'PROFESSIONAL',['EDUCATION']]     = 3


In [27]:
print(df['EDUCATION'].value_counts())
df['EDUCATION'] = df['EDUCATION'].astype(int)
df.info()

EDUCATION
3    18931
2    11703
1     9532
4     1898
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 43 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   pct_tl_open_L6M            42064 non-null  float64
 1   pct_tl_closed_L6M          42064 non-null  float64
 2   Tot_TL_closed_L12M         42064 non-null  int64  
 3   pct_tl_closed_L12M         42064 non-null  float64
 4   Tot_Missed_Pmnt            42064 non-null  int64  
 5   CC_TL                      42064 non-null  int64  
 6   Home_TL                    42064 non-null  int64  
 7   PL_TL                      42064 non-null  int64  
 8   Secured_TL                 42064 non-null  int64  
 9   Unsecured_TL               42064 non-null  int64  
 10  Other_TL                   42064 non-null  int64  
 11  Age_Oldest_TL              42064 non-null  int64  
 12  Age_Newest_TL         

In [29]:
df_encoded = pd.get_dummies(df , columns = ['MARITALSTATUS' , 'GENDER' , 
                                            'last_prod_enq2' , 'first_prod_enq2'])
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 55 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   pct_tl_open_L6M               42064 non-null  float64
 1   pct_tl_closed_L6M             42064 non-null  float64
 2   Tot_TL_closed_L12M            42064 non-null  int64  
 3   pct_tl_closed_L12M            42064 non-null  float64
 4   Tot_Missed_Pmnt               42064 non-null  int64  
 5   CC_TL                         42064 non-null  int64  
 6   Home_TL                       42064 non-null  int64  
 7   PL_TL                         42064 non-null  int64  
 8   Secured_TL                    42064 non-null  int64  
 9   Unsecured_TL                  42064 non-null  int64  
 10  Other_TL                      42064 non-null  int64  
 11  Age_Oldest_TL                 42064 non-null  int64  
 12  Age_Newest_TL                 42064 non-null  int64  
 13  t

## Model fitting
using 3 models 
1) RandomForest
2) Xgboost
3) Decision Tree

In [33]:
#) 1. Random Forest

y = df_encoded['Approved_Flag']
x = df_encoded.drop(['Approved_Flag'] , axis = 1)


x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 42)

rf_classifier = RandomForestClassifier(n_estimators = 200 , random_state = 42)
rf_classifier.fit(x_train , y_train)
y_pred = rf_classifier.predict(x_test)



accuracy = accuracy_score(y_test , y_pred)
print()
print(f'Accuracy : { accuracy}')
print()

precision , recall , f1_score ,_ = precision_recall_fscore_support(y_test , y_pred)



for i , v in enumerate( ['p1' , 'p2' , 'p3' , 'p4']):
    print(f'Class {v}:')
    print(f'Precision: {precision[i]}')
    print(f'Recall : {recall[i]}')
    print(f'F1 score : {f1_score[i]}')
    print()



Accuracy : 0.7636990372043266

Class p1:
Precision: 0.8370457209847597
Recall : 0.7041420118343196
F1 score : 0.7648634172469202

Class p2:
Precision: 0.7957519116397621
Recall : 0.9282457879088206
F1 score : 0.856907593778591

Class p3:
Precision: 0.4423380726698262
Recall : 0.21132075471698114
F1 score : 0.28600612870275793

Class p4:
Precision: 0.7178502879078695
Recall : 0.7269193391642371
F1 score : 0.7223563495895703



 As Class 3 values in dataset overlaps with class 2 and class 4 some times , the Score value for class 3 is not high ,
So even after using model ,if outcome is P3 Class then it will need to careful consideration for actions to choose 

In [45]:
# 2. xgboost

import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

xgb_classifier = xgb.XGBClassifier(objective='multi:softmax',  num_class=4)



y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)




xgb_classifier.fit(x_train, y_train)
y_pred = xgb_classifier.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print ()
print(f'Accuracy: {accuracy:.2f}')
print ()

precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()



Accuracy: 0.78

Class p1:
Precision: 0.823906083244397
Recall: 0.7613412228796844
F1 Score: 0.7913890312660175

Class p2:
Precision: 0.8255418233924413
Recall: 0.913577799801784
F1 Score: 0.8673315769665035

Class p3:
Precision: 0.4756380510440835
Recall: 0.30943396226415093
F1 Score: 0.37494284407864653

Class p4:
Precision: 0.7342386032977691
Recall: 0.7356656948493683
F1 Score: 0.7349514563106796



In [49]:
# 3) Decision Tree

from sklearn.tree import DecisionTreeClassifier

y = df_encoded['Approved_Flag']
x = df_encoded.drop(['Approved_Flag'] , axis = 1)

x_train ,x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state =42)

dtr = DecisionTreeClassifier( max_depth = 20 , min_samples_split = 10)
dtr.fit( x_train , y_train)
y_pred = dtr.predict( x_test)

accuracy = accuracy_score(y_test , y_pred)
print()
print(f'Accuracy : {accuracy :.2f}')
print()

precision , recall , f1_score  , _ = precision_recall_fscore_support(y_test , y_pred)

for i , v in enumerate(['p1' , 'p2' , 'p3' , 'p4' ]):
    print(f'Class : {v}')
    print(f'Precision : {precision[i]}')
    print(f'Recall : {recall[i]}')
    print(f'F1 score : {f1_score}')
    print()


Accuracy : 0.71

Class : p1
Precision : 0.7227138643067846
Recall : 0.7248520710059172
F1 score : [0.72378139 0.81699282 0.33461243 0.63267327]

Class : p2
Precision : 0.8106947697111632
Recall : 0.8233894945490585
F1 score : [0.72378139 0.81699282 0.33461243 0.63267327]

Class : p3
Precision : 0.3403590944574551
Recall : 0.3290566037735849
F1 score : [0.72378139 0.81699282 0.33461243 0.63267327]

Class : p4
Precision : 0.644803229061554
Recall : 0.6209912536443148
F1 score : [0.72378139 0.81699282 0.33461243 0.63267327]



We have XGBClassifier with most accuracy , so will consider it
and further fine tune it 

### Fine-tuning XGBClassifier

In [52]:
# applying standard scaler

from sklearn.preprocessing import StandardScaler

columns_to_be_scaled = ['Age_Oldest_TL','Age_Newest_TL','time_since_recent_payment',
'max_recent_level_of_deliq','recent_level_of_deliq',
'time_since_recent_enq','NETMONTHLYINCOME','Time_With_Curr_Empr']

for i in columns_to_be_scaled :
    column_data = df_encoded[i].values.reshape(-1 , 1)
    scaler = StandardScaler()
    scaled_column = scaler.fit_transform(column_data)
    df_encoded[i] = scaled_column

In [53]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

xgb_classifier = xgb.XGBClassifier(objective='multi:softmax',  num_class=4)



y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)




xgb_classifier.fit(x_train, y_train)
y_pred = xgb_classifier.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')


precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()
    
    

Accuracy: 0.78
Class p1:
Precision: 0.823906083244397
Recall: 0.7613412228796844
F1 Score: 0.7913890312660175

Class p2:
Precision: 0.8255418233924413
Recall: 0.913577799801784
F1 Score: 0.8673315769665035

Class p3:
Precision: 0.4756380510440835
Recall: 0.30943396226415093
F1 Score: 0.37494284407864653

Class p4:
Precision: 0.7342386032977691
Recall: 0.7356656948493683
F1 Score: 0.7349514563106796



In [59]:
# hyperparameter tuning xgboost

from sklearn.model_selection import GridSearchCV
x_train , x_test , y_train , y_test = train_test_split( x , y_encoded , test_size = 0.2 , random_state =42)


xgb_model = xgb.XGBClassifier(objective  = 'multi:softmax' , num_class = 4)

# parameter grid for tuning

param_grid = {
    'n_estimators' : [50 , 100 , 200 ],
    'max_depth' : [3,5,7] ,
    'learning_rate' :[ 0.01 , 0.1 , 0.2]
}

grid_search = GridSearchCV( estimator = xgb_model , param_grid = param_grid  ,cv = 5 , scoring = 'accuracy' , n_jobs = -1)
grid_search.fit(x_train , y_train)

# best hyperparameters 
print("Best Hyperparameter : " , grid_search.best_params_)

#evalutation of best parameter
best_model = grid_search.best_estimator_
accuracy = best_model.score(x_test ,y_test)
print('Test Accuracy : ' , accuracy)



Best Hyperparameter :  {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200}
Test Accuracy :  0.7811719957209081


In [ ]:
# based of risk appetite , recommed the class

### Extra Analysis

In [73]:
# detailed analyis of parameter evalution with extra parameter 

param_grid = {
    'colsample_bytree' :[0.1  , 0.3 , 0.5 , 0.7 , 0.9],
    'learning_rate' : [0.001 , 0.01 , 0.1 ,1],
    'max_depth' : [3,5,8,10],
    'alpha' : [1 , 10 , 100],
    'n_estimators' : [10 , 50 , 100]
}

index = 0

answer_grid = {
    'combination' : [],
    'train_Accuracy' : [] ,
    'test_Accuracy' : [] ,
    'colsample_bytree' :[] , 
    'learning_rate' :[] , 
    'max_depth' :[],
    'alpha' : [],
    'n_estimators' : []
}

# looping thru each combination of hyperparameter and storing values 

for colsample_bytree in param_grid['colsample_bytree']:
    for learning_rate in param_grid['learning_rate']:
        for max_depth in param_grid['max_depth']:
            for alpha in param_grid['alpha']:
                for n_estimators in param_grid['n_estimators']:

                    index = index + 1

                    # defining model
                    model = xgb.XGBClassifier(objective = 'multi:softmax',
                                             num_class = 4 , 
                                             colsample_bytree = colsample_bytree,
                                             learning_rate = learning_rate,
                                             max_depth = max_depth,
                                             alpha = alpha ,
                                             n_estimators = n_estimators)

                    y = df_encoded['Approved_Flag']
                    x = df_encoded.drop(['Approved_Flag'] , axis = 1)

                    label_encoder = LabelEncoder()
                    y_encoded = label_encoder.fit_transform(y)

                    x_train , x_test , y_train , y_test = train_test_split(x , y_encoded , test_size = 0.2 , random_state = 42)

                    model.fit(x_train , y_train)

                    #prediction
                    y_pred_train = model.predict(x_train)
                    y_pred_test = model.predict(x_test)

                    #accuracy
                    train_accuracy = accuracy_score(y_train , y_pred_train)
                    test_accuracy = accuracy_score(y_test , y_pred_test)

                    # appending all into list
                    answer_grid['combination'].append(index)
                    answer_grid['train_Accuracy'].append(train_accuracy)
                    answer_grid['test_Accuracy'].append(test_accuracy)
                    answer_grid['colsample_bytree'].append(colsample_bytree)
                    answer_grid['learning_rate'].append(learning_rate)
                    answer_grid['max_depth'].append(max_depth)
                    answer_grid['alpha'].append(alpha)
                    answer_grid['n_estimators'].append(n_estimators)

                    # # printing combinations
                    # print(f'Combination : {index}')
                    # print(f"colsample_bytree: {colsample_bytree}, learning_rate: {learning_rate}, max_depth: {max_depth}, alpha: {alpha}, n_estimators: {n_estimators}")
                    # print(f'Train Accuracy : {train_accuracy:.2f}')
                    # print(f'Test Accuracy : {test_accuracy:.2f}')
                    # print('-'*15)




In [75]:
combinations_df = pd.DataFrame(answer_grid)
combinations_df

,combination,train_Accuracy,test_Accuracy,colsample_bytree,learning_rate,max_depth,alpha,n_estimators
0,1,0.606431,0.599667,0.1,0.001,3,1,10
1,2,0.606460,0.599667,0.1,0.001,3,1,50
2,3,0.606431,0.599667,0.1,0.001,3,1,100
3,4,0.606431,0.599667,0.1,0.001,3,10,10
4,5,0.606431,0.599667,0.1,0.001,3,10,50
...,...,...,...,...,...,...,...,...
715,716,0.939140,0.767027,0.9,1.000,10,10,50
716,717,0.962379,0.766076,0.9,1.000,10,10,100
717,718,0.784969,0.770118,0.9,1.000,10,100,10
718,719,0.788179,0.773565,0.9,1.000,10,100,50


In [77]:
combinations_df.to_csv('combi.csv' , index = False)